In [6]:
from pathlib import Path
from time import perf_counter

import mlx.core as mx
import numpy as np
import torch
from cut_cross_entropy import linear_cross_entropy
from mlx_vlm import load, stream_generate
from mlx_vlm.prompt_utils import apply_chat_template
from mlx_vlm.utils import load_config
from torch import nn
from torch.nn import functional as F
from transformers.image_utils import load_image


HIDDEN_SIZE = 576
VOCAB_SIZE = 100_352
NUM_PREDICTIONS = 4

In [ ]:
class LatentDraft(nn.Module):
    """Two causal Transformer blocks with parallel future-token heads."""

    def __init__(self, context_length: int) -> None:
        super().__init__()
        self.positions = nn.Embedding(context_length, HIDDEN_SIZE)
        block = nn.TransformerEncoderLayer(
            d_model=HIDDEN_SIZE,
            nhead=9,
            dim_feedforward=1536,
            dropout=0.0,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(block, num_layers=1)
        self.norm = nn.LayerNorm(HIDDEN_SIZE)
        self.future_heads = nn.Linear(HIDDEN_SIZE, NUM_PREDICTIONS * HIDDEN_SIZE)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        sequence_length = hidden_states.shape[1]
        positions = torch.arange(sequence_length, device=hidden_states.device)
        causal_mask = torch.triu(
            torch.ones(sequence_length, sequence_length, device=hidden_states.device, dtype=torch.bool),
            diagonal=1,
        )
        hidden_states = self.norm(
            self.transformer(
                hidden_states + self.positions(positions), mask=causal_mask, is_causal=True
            )
        )
        return self.future_heads(hidden_states).unflatten(-1, (NUM_PREDICTIONS, HIDDEN_SIZE))


def load_trace(path: Path, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Load a trace written by fastdocling-extract (safetensors, causal prefill)."""
    trace = mx.load(str(path))
    if "token_ids" not in trace or "prompt_len" not in trace:
        raise ValueError(f"{path} is missing token_ids or prompt_len; regenerate it with fastdocling-extract")

    hidden_states = torch.from_numpy(np.asarray(trace["last_hidden_state"].astype(mx.float32)))
    token_ids = torch.from_numpy(np.asarray(trace["token_ids"])).long()
    prompt_len = int(np.asarray(trace["prompt_len"]).item())

    # Train only on generated DocTags, not the fixed multimodal prompt prefix.
    return hidden_states[prompt_len - 1 :].to(device), token_ids[prompt_len - 1 :].to(device)

In [ ]:
trace_paths = sorted(Path("../data/traces").glob("*.safetensors"))
if not trace_paths:
    raise FileNotFoundError("run `fastdocling-extract data/images data/traces` first")
steps = 250
context_length = 256
learning_rate = 3e-4
output = "latent_draft.pt"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
traces = [load_trace(path, device) for path in trace_paths]
available_tokens = sum(hidden_states.shape[0] - NUM_PREDICTIONS for hidden_states, _ in traces)
if available_tokens < context_length + NUM_PREDICTIONS:
    raise ValueError("The traces are shorter than --context-length")
if available_tokens < 100_000:
    print(f"Warning: only {available_tokens:,} training tokens; use many more pages for a useful draft model.")

# Reuse Granite's frozen vocabulary projection, so 576-wide outputs become token logits.
target_model, target_processor = load("ibm-granite/granite-docling-258M-mlx")
lm_head = torch.from_numpy(
    np.asarray(target_model.language_model.lm_head.weight.astype(mx.float32))
).to(device)
if lm_head.shape != (VOCAB_SIZE, HIDDEN_SIZE):
    raise ValueError(f"Unexpected Granite LM head shape: {tuple(lm_head.shape)}")

draft = torch.compile(LatentDraft(context_length).to(device), mode="reduce-overhead")
optimizer = torch.optim.AdamW(draft.parameters(), lr=learning_rate)
draft.train()
print(f"Training on {available_tokens:,} tokens with {sum(p.numel() for p in draft.parameters()):,} draft parameters")

In [11]:
# Exclude the first optimizer step, which compiles both torch.compile graphs.
training_started_at = None
training_tokens = 0
for step in range(1, steps + 1):
    hidden_states, token_ids = traces[torch.randint(len(traces), ()).item()]
    start = torch.randint(hidden_states.shape[0] - context_length - NUM_PREDICTIONS + 1, ()).item()
    inputs = hidden_states[start : start + context_length].unsqueeze(0)
    target_hidden = torch.stack(
        [hidden_states[start + offset : start + context_length + offset] for offset in range(1, NUM_PREDICTIONS + 1)],
        dim=1,
    ).unsqueeze(0)
    target_tokens = torch.stack(
        [token_ids[start + offset : start + context_length + offset] for offset in range(1, NUM_PREDICTIONS + 1)],
        dim=1,
    ).unsqueeze(0)

    predicted_hidden = draft(inputs)
    # Apple CCE avoids materializing the 100,352-token logit matrix per head.
    token_loss = sum(
        linear_cross_entropy(
            predicted_hidden[:, :, offset].flatten(0, 1),
            lm_head,
            target_tokens[:, :, offset].flatten(),
            impl="torch_compile",
        )
        for offset in range(NUM_PREDICTIONS)
    ) / NUM_PREDICTIONS
    hidden_loss = F.mse_loss(predicted_hidden, target_hidden)
    loss = token_loss + 0.1 * hidden_loss

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step == 1:
        if device.type == "mps":
            torch.mps.synchronize()
        training_started_at = perf_counter()
    else:
        training_tokens += context_length

    if step == 1 or step % 100 == 0:
        print(f"step={step} loss={loss.item():.4f} token_loss={token_loss.item():.4f}")

torch.save(
    {"state_dict": draft.state_dict(), "context_length": context_length},
    output,
)
print(f"Saved {output}")
if device.type == "mps":
    torch.mps.synchronize()
training_seconds = perf_counter() - training_started_at
print(f"Training throughput: {training_tokens / training_seconds:.1f} tokens/s ({training_seconds:.2f}s total)")   

W0914 01:30:31.462000 65598 torch/_inductor/cudagraph_utils.py:401] [3/2] [__cudagraphs] skipping cudagraphs due to skipping cudagraphs due to multiple devices: device(type='mps', index=0)


step=1 loss=3.2626 token_loss=2.9614
step=100 loss=2.6193 token_loss=2.3149
step=200 loss=1.1658 token_loss=0.8669
Saved latent_draft.pt
Training throughput: 1938.6 tokens/s (32.88s total)


In [12]:
# Measure raw target decoding on the same final page used to create the trace.
TEST_TOKENS = 128
test_image = load_image("resnet_first_5_pages_png/page_05.png")
target_config = load_config("ibm-granite/granite-docling-258M-mlx")
test_prompt = apply_chat_template(
    target_processor, target_config, "Convert this page to docling.", num_images=1
)
# Warm model wiring and image prefill; measure steady-state decoding below.
for _ in stream_generate(
    target_model, target_processor, test_prompt, [test_image],
    max_tokens=1, temp=0.0, verbose=False,
):
    pass
baseline_tokens = 0
baseline_decode_tps = 0.0
for result in stream_generate(
    target_model, target_processor, test_prompt, [test_image],
    max_tokens=TEST_TOKENS, temp=0.0, verbose=False,
):
    baseline_tokens = result.generation_tokens
    baseline_decode_tps = result.generation_tps
baseline_decode_seconds = baseline_tokens / baseline_decode_tps
print(f"Target decode baseline: {baseline_decode_tps:.1f} tokens/s")

# Teacher-forced oracle evaluation: the target trace verifies each draft proposal.
# A verifier batch implementation is still needed for this estimate to become an
# end-to-end speculative decoder benchmark.
draft.eval()
test_hidden, test_ids = traces[0]
test_tokens = min(TEST_TOKENS, test_hidden.shape[0] - 1)
cursor = 0
verification_calls = 0
accepted_tokens = 0
# Compile the one-token proposal shape before starting the draft timer.
with torch.inference_mode():
    draft(test_hidden[:1].unsqueeze(0))
if device.type == "mps":
    torch.mps.synchronize()
draft_started_at = perf_counter()
with torch.inference_mode():
    while cursor < test_tokens:
        rollout = test_hidden[cursor : cursor + 1].unsqueeze(0)
        proposed = []
        predicted_hidden = draft(rollout[:, -context_length:])[:, -1]
        proposed = F.linear(predicted_hidden, lm_head).argmax(dim=-1).tolist()[0]
        proposed = proposed[: min(NUM_PREDICTIONS, test_tokens - cursor)]

        verification_calls += 1
        accepted = 0
        for proposed_token, target_token in zip(proposed, test_ids[cursor + 1 :]):
            if proposed_token != target_token.item():
                break
            accepted += 1
        accepted_tokens += accepted
        cursor += min(accepted + 1, test_tokens - cursor)
if device.type == "mps":
    torch.mps.synchronize()
draft_seconds = perf_counter() - draft_started_at
average_accepted = accepted_tokens / verification_calls
# This is an upper-bound estimate: it assumes each target verification batch costs
# one normal decode step. It includes the measured PyTorch draft rollout time.
estimated_seconds = verification_calls / baseline_decode_tps + draft_seconds
estimated_speedup = baseline_decode_seconds / estimated_seconds
print(f"Draft rollout: {test_tokens / draft_seconds:.1f} proposed tokens/s")
print(f"Acceptance: {accepted_tokens}/{test_tokens} tokens, {average_accepted:.2f} accepted/verification")
print(f"Estimated speculative speedup: {estimated_speedup:.2f}x (not an end-to-end verifier benchmark)")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Target decode baseline: 188.0 tokens/s
Draft rollout: 509.2 proposed tokens/s
Acceptance: 67/128 tokens, 1.10 accepted/verification
Estimated speculative speedup: 1.18x (not an end-to-end verifier benchmark)
